[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1/blob/main/diego_bispo.ipynb)

# Prototipo autonomo do benchmark OAB

Este notebook utiliza o arquivo `curadorias.csv` na raiz do projeto e os modulos Python posicionados na raiz do projeto, mantendo a pasta `artefatos_estudo/` apenas como referencia metodologica.

## Fluxo
1. Configuracao do ambiente
2. Preparacao dos dados
3. Download dos modelos
4. Geracao ou reaproveitamento das respostas
5. Avaliacao das objetivas
6. Avaliacao discursiva estruturada com base no novo gabarito
7. Consolidado executivo


## 1. Configuracao do ambiente

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas numpy tqdm seaborn matplotlib huggingface_hub sentencepiece scipy sentence-transformers


In [ ]:
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Markdown, display

from configuracoes import criar_configuracao_padrao
from pipeline import (
    baixar_modelos_requeridos,
    executar_avaliacao_discursiva,
    executar_avaliacao_objetiva,
    executar_consolidado,
    gerar_ou_carregar_respostas,
    preparar_conjunto_dados,
)

RAIZ_PROJETO = Path.cwd()
CONFIGURACAO = criar_configuracao_padrao(RAIZ_PROJETO)
CONFIGURACAO.preparar_ambiente()

print('Diretorio de trabalho:', RAIZ_PROJETO)
print('curadorias.csv disponivel?', CONFIGURACAO.curadorias_csv.exists())
print('GPU:', torch.cuda.get_device_name(0))
print('Saidas em:', CONFIGURACAO.diretorio_saida)


## 2. Preparacao dos dados

In [ ]:
df_curadoria, df_objetivas, df_discursivas = preparar_conjunto_dados(CONFIGURACAO)

print('Linhas curadas:', len(df_curadoria))
print('Objetivas:', len(df_objetivas))
print('Discursivas:', len(df_discursivas))

display(df_objetivas[[
    'ID da Quest?o', 'texto_questao', 'gabarito_oficial', 'nivel_dificuldade', 'disciplina'
]].head())

display(df_discursivas[[
    'ID da Quest?o', 'Tipo Quest?o', 'texto_questao', 'pontuacao_total', 'nivel_dificuldade', 'disciplina'
]].head())


## 3. Download dos modelos

In [ ]:
print('Baixando modelos...')
baixar_modelos_requeridos(CONFIGURACAO)


## 4. Geracao ou reaproveitamento das respostas

In [ ]:
df_respostas_objetivas, df_respostas_discursivas = gerar_ou_carregar_respostas(
    CONFIGURACAO,
    df_objetivas,
    df_discursivas,
)

display(df_respostas_objetivas.head())
display(df_respostas_discursivas.head())


## 5. Avaliacao das objetivas

In [ ]:
df_objetivas_detalhe, benchmark_objetivas = executar_avaliacao_objetiva(
    CONFIGURACAO,
    df_respostas_objetivas,
)

melhor_objetiva = benchmark_objetivas.iloc[0]
display(Markdown(
    f"**Melhor modelo nas objetivas:** `{melhor_objetiva['modelo']}` com acuracia de **{melhor_objetiva['acuracia_objetivas']:.2%}** "
    f"({int(melhor_objetiva['acertos_objetivas'])}/{int(melhor_objetiva['total_objetivas'])})"
))

display(benchmark_objetivas.round(4))


## 6. Avaliacao discursiva estruturada

A analise qualitativa anterior foi substituida por uma correcao estruturada baseada no novo `gabarito`, utilizando os `criterios` e seus `componentes` semanticos e legislativos. A parte semantica combina Legal-SBERT com NLI, conforme consolidado em `artefatos_estudo/Metricas_Escolhidas.ipynb`.


In [ ]:
df_discursivas_detalhe, benchmark_discursivas, df_composicao_discursiva = executar_avaliacao_discursiva(
    CONFIGURACAO,
    df_respostas_discursivas,
)

melhor_discursiva = benchmark_discursivas.iloc[0]
display(Markdown(
    f"**Melhor modelo nas discursivas:** `{melhor_discursiva['modelo']}` com aproveitamento de "
    f"**{melhor_discursiva['aproveitamento_geral']:.2%}** e **{melhor_discursiva['nota_total']:.2f}** pontos em "
    f"**{melhor_discursiva['pontuacao_total_disc']:.2f}** possiveis."
))

display(benchmark_discursivas.round(4))
display(df_discursivas_detalhe.head().round(4))
display(df_composicao_discursiva.head().round(4))


## 7. Consolidado executivo

In [ ]:
benchmark_consolidado = executar_consolidado(
    CONFIGURACAO,
    df_objetivas_detalhe,
    benchmark_objetivas,
    df_discursivas_detalhe,
    benchmark_discursivas,
    df_composicao_discursiva,
)

display(benchmark_consolidado.round(4))
